In [ ]:
import json
from app.src.triage_graph import llm_triage, generate_graph
from app.src.agent_member_info import SearchMemberInfoAgent
from app.src.agent_search_knowleadge import SearchKnowledgeBaseAgent
from langgraph.types import Command

cases = [json.loads(l) for l in open("eval_cases.jsonl", encoding="utf-8")]

/Users/gasparcartasso/Desktop/agent_developer_tp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
super_agent = generate_graph()
for case in cases:
    if case.get("category") == "triage":
        result = super_agent.invoke(
                {"email_body": case['email_body'], "user": f"test_user_{case['id']}",
                 "thread": case['id'], "project": "kb"},
                config={"configurable": {"thread_id": case['id']}},
            )
        print(f"Case {case['id']} {case['expected'].get('is_question_about_policy')} triage result: {result.get('is_question_about_policy')}")


Case tri-01 True triage result: True
Case tri-02 False triage result: False
Case tri-03 True triage result: True
Case tri-04 False triage result: True
Case tri-05 True triage result: True
Case tri-06 True triage result: True
Case tri-07 True triage result: True
Case tri-08 False triage result: False
Case tri-09 False triage result: True
Case tri-10 True triage result: True


In [2]:
search_member_info = SearchMemberInfoAgent()
for case in cases:    
    if case.get('category')=="member_lookup":
        result =search_member_info.ask(case['email_body'], user=f"test_user_{case['id']}", thread=case['id'], project="kb")
        print(f"Case {case['id']} {case['expected']} member lookup result: {result.get('final_answer')} tool/s: {result.get('tool_called')}")

Case mem-01 {'product': 'Basic Accident Only Hospital $750 Excess', 'must_contain': ['Basic Accident Only Hospital $750 Excess']} member lookup result: The member number 14265799 has the following product:

- **Product**: Basic Accident Only Hospital $750 Excess tool/s: None
Case mem-02 {'product': 'Standard Visitors Cover', 'must_contain': ['Standard Visitors Cover']} member lookup result: The member number 13356886 has the following product:

- **Product:** Standard Visitors Cover tool/s: None
Case mem-03 {'product': 'Gold Comprehensive Hospital $750 Excess', 'must_contain': ['Gold Comprehensive Hospital $750 Excess']} member lookup result: The member number 39958838 is associated with the product **Gold Comprehensive Hospital $750 Excess**. tool/s: None
Case mem-04 {'product': 'Bupa Employee Bronze Plus Hospital', 'must_contain': ['Bupa Employee Bronze Plus Hospital']} member lookup result: The member number 23756669 is associated with the product **Bupa Employee Bronze Plus Hospita

In [3]:
search_kb_info = SearchKnowledgeBaseAgent()
for case in cases:    
    if case.get('category') in ["kb_grounded","kb_abstain"]:
        result =search_kb_info.ask(case['email_body'], user=f"test_user_{case['id']}", thread=case['id'], project="kb")
        print(f"Case {case['id']} {case['expected']} member lookup result: {result.get('final_answer')} tool/s: {result.get('tool_called')}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5926.76it/s]


Case kb-01 {'must_contain_any': ['12 months'], 'must_not_contain': []} member lookup result: The waiting period for pregnancy and birth (obstetrics) under the Gold Complete Hospital cover is 12 months. This information is consistent across multiple documents [bupa cover#182], [bupa cover#147], and [bupa cover#148]. tool/s: None
Case kb-02 {'must_contain_any': ['12 months'], 'must_not_contain': []} member lookup result: The waiting period for pre-existing conditions with Gold Complete Hospital cover is **12 months** [bupa cover#182], [bupa cover#297], [bupa cover#147]. tool/s: None
Case kb-03 {'must_contain_any': ['not covered', 'excluded', 'no benefit'], 'must_not_contain': []} member lookup result: - **Cosmetic surgery is excluded** under the Basic Accident Only Hospital cover. This is explicitly stated in [bupa cover#37] and [bupa cover#47]. tool/s: None
Case kb-04 {'must_contain_any': ['$500', '$750'], 'must_not_contain': ['$250', 'nil excess']} member lookup result: For the **Basic